# 06 · AI Smog Explainer: bilingual advisories from the model's numbers

<a href="https://colab.research.google.com/github/FarhanRajputFelix/saans/blob/main/notebooks/06_llm_advisories.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The second LLM use in SAANS is in the product. For **each city and day**, the LLM receives the forecast, the AQI, the hazard probability and the top three drivers from SHAP, and writes a three-line advisory in **English and Urdu**. The code lives in `saans/advisory.py`, shared with the live forecast (notebook 07) and the dashboard.

Guardrails:
1. **The LLM never predicts.** It only rewords numbers produced by the models.
2. **Automatic number check.** Every number in its answer (English and Urdu, Urdu digits included) must be one we supplied, or a fixed term such as PM2.5 or N95. Answers that fail are discarded.
3. **Template fallback.** A rule-based bilingual template is used when the LLM fails the check or is unavailable.
4. **Cached once per city per day** (`app/data/advisories.json`), so the dashboard needs no LLM key.

**Setup.** Runs the same way locally and in Google Colab. In Colab it clones the repo (which includes the cleaned data) and reads API keys from Colab's *Secrets* panel (key icon on the left).

In [1]:
import os, sys, subprocess
if "google.colab" in sys.modules:
    if not os.path.exists("/content/saans"):
        subprocess.run(["git", "clone", "-q", "https://github.com/FarhanRajputFelix/saans.git", "/content/saans"], check=True)
    os.chdir("/content/saans")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "shap", "python-dotenv"], check=True)
    from google.colab import userdata
    for k in ["OPENAQ_API_KEY", "FIRMS_MAP_KEY", "GEMINI_API_KEY", "GROQ_API_KEY"]:
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # work from the repo root so `saans` and `data/` resolve
sys.path.insert(0, os.getcwd())

In [2]:
import json
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

from saans.config import ROOT
from saans import advisory

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 200)
DASH = ROOT / "app" / "data"
daily = pd.read_parquet(DASH / "forecasts_daily.parquet")
daily["date"] = daily.time_pkt.dt.tz_convert("Asia/Karachi").dt.date.astype(str)
print(f"{len(daily)} city-days need an advisory (every daily forecast in both test seasons)")

# The proposal budgets "only a few dozen LLM calls" for the demo. The LLM writes the days that matter most:
# every city on the dashboard's two default dates and each city's three worst forecast days per season.
# Every other day gets the rule-based bilingual template, labelled as such on the dashboard.
mid_2025 = sorted(daily[daily.season == "2025-26"].date.unique())
use_llm = daily.date.isin(["2024-11-06", mid_2025[len(mid_2025) // 2]])
for _, g in daily.groupby(["season", "city"]):
    use_llm.loc[g.nlargest(3, "forecast_24h").index] = True
daily["use_llm"] = use_llm
print(f"LLM advisories: {int(use_llm.sum())} city-days; template: {int((~use_llm).sum())}")

930 city-days need an advisory (every daily forecast in both test seasons)
LLM advisories: 32 city-days; template: 898


## The prompt

One example, filled in with the model's numbers:

In [3]:
ex = daily[(daily.city == "lahore") & (daily.date == "2024-11-06")].iloc[0]
print(advisory.PROMPT.format(**advisory.facts(ex.city, ex.date, ex.forecast_24h, ex.hazard_prob, ex.top_drivers)))

You are the "AI Smog Explainer" of SAANS, a smog forecasting system for Pakistan.
Write a health advisory for residents of Lahore for the next 24 hours (forecast issued 2024-11-06, 08:00 Pakistan time).

Use ONLY these numbers, exactly as written. Do not calculate, round differently or invent any other number:
- forecast 24-hour mean PM2.5: 383 micrograms per cubic metre
- US EPA AQI: 616 (Hazardous)
- chance that the next 24 hours are hazardous (AQI above 300): 93%
- main reasons, from the model's explanation: average pm2.5 over the last 24 hours; pm2.5 right now; upwind crop fires, last 24 hours

Write exactly three short lines in plain English:
1. the forecast and the AQI;
2. why, naming the main reasons in everyday words;
3. what people should do, with care for children, the elderly and people with asthma.
Then write the same three lines in natural, simple Urdu (Urdu script).

Return JSON: {"en": "<three lines separated by \n>", "ur": "<three lines separated by \n>"}


## Write all advisories

Four requests run in parallel. Gemini 3.5 Flash-Lite answers first; when its free daily quota runs out the client moves to Gemini 3.1 Flash-Lite, then Groq (see `saans/llm.py`).

In [4]:
def one(r):
    return f"{r.city}|{r.date}", advisory.write(r.city, r.date, r.forecast_24h, r.hazard_prob, r.top_drivers, use_llm=r.use_llm)

with ThreadPoolExecutor(max_workers=4) as pool:
    advisories = dict(pool.map(one, daily.itertuples()))
(DASH / "advisories.json").write_text(json.dumps(advisories, ensure_ascii=False, indent=1), encoding="utf-8")
sources = pd.Series([a["source"] for a in advisories.values()]).value_counts()
print(f"{len(advisories)} advisories written and cached")
sources.rename("advisories").to_frame()

930 advisories written and cached


,advisories
template,899
gemini:gemini-3.5-flash-lite,31


`template` rows are the days outside the LLM budget, plus any day where the LLM was unavailable or its answer failed the number check (the guardrail then substitutes the rule-based text).

## Example: Lahore, 6 November 2024

In [5]:
a = advisories["lahore|2024-11-06"]
print(f"[{a['source']}]\n{a['en']}\n\n{a['ur']}")

[gemini:gemini-3.5-flash-lite]
Lahore's forecast 24-hour mean PM2.5 is 383 micrograms per cubic metre, with a US EPA AQI of 616 (Hazardous), and a 93% chance that the next 24 hours are hazardous (AQI above 300).
This is mainly driven by average pm2.5 over the last 24 hours; pm2.5 right now; upwind crop fires, last 24 hours.
Everyone should stay indoors, and children, the elderly, and people with asthma need extra care and must avoid going outside.

لاہور میں اگلے 24 گھنٹوں کے دوران PM2.5 کی اوسط پیشگوئی 383 مائیکرو گرام فی مکعب میٹر ہے، US EPA AQI 616 (خطرناک) ہے، اور اگلے 24 گھنٹوں کے خطرناک (AQI 300 سے اوپر) ہونے کا امکان 93% ہے۔
اس کی بنیادی وجوہات پچھلے 24 گھنٹوں میں PM2.5 کی اوسط، اس وقت کا PM2.5، اور پچھلے 24 گھنٹوں میں ہوا کی سمت کے رخ پر فصلوں کی باقیات جلانا ہیں۔
سب لوگ گھروں کے اندر رہیں، جبکہ بچوں، بزرگوں اور دمہ کے مریضوں کا خاص خیال رکھیں اور انہیں باہر جانے سے گریز کریں۔
